# P03-P10 · Conciliar Metro y medir correspondencias con buses

> **¿Qué estaciones de Metro tienen buses a 200 m y cuánto depende la respuesta de la fuente?**

GTFS Red Movilidad y el archivo público de estaciones de Metro para carga bip!.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/correspondencia_metro/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# 1. Detectar el entorno y obtener una copia limpia del repositorio.
# Esta celda también funciona localmente si se abre el notebook dentro del repo.
import os
import platform
import subprocess
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
if not Path('pyproject.toml').exists():
    destino_repo = Path('lab-dtpm')
    if not destino_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git', str(destino_repo)], check=True)
    os.chdir(destino_repo)

ROOT = Path.cwd().resolve()
PAQUETES = ['pandas>=2.2', 'numpy>=1.26', 'openpyxl', 'XlsxWriter', 'matplotlib', 'polars>=1.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PAQUETES], check=True)
print('Entorno:', 'Google Colab' if EN_COLAB else 'Python local/Jupyter')
print('Python:', platform.python_version())
print('Repositorio:', ROOT)


In [ ]:
# 2. Importar y registrar versiones: evita resultados imposibles de auditar.
import matplotlib
import numpy as np
import pandas as pd
import polars as pl

print({'pandas': pd.__version__, 'polars': pl.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__})


## Alternativas para integrar los datos

La vía reproducible es clonar GitHub y ejecutar `scripts/fetch_data.py`, como hace la siguiente celda. Si la red institucional lo impide, consulte la [guía portable](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/docs/ejecucion-portable.md): explica carga desde el equipo, montaje de Google Drive, descarga directa y validación SHA-256. No cambie la fuente por datos simulados sin declararlo.


In [ ]:
# 3. Obtener las fuentes originales declaradas por el caso.
# Si la fuente externa falla, el error se conserva: no se sustituyen datos
# reales por sintéticos. Los CSV versionados permiten ejecutar el laboratorio
# comparativo, pero no reemplazan la adquisición del caso completo.
fuentes = ['gtfs-red-movilidad', 'bip-metro', 'bip-puntos', 'bip-centros-alto-estandar', 'bip-centros-estandar', 'bip-retail']
for fuente in fuentes:
    try:
        subprocess.run([sys.executable, 'scripts/fetch_data.py', fuente], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'No se pudo obtener {fuente}. Revise la URL, la red y las alternativas de carga descritas arriba.'
        ) from exc


## 1 · El problema

Dos fuentes públicas describen las estaciones de Metro con granos diferentes: el GTFS
tiene 126 estaciones y el archivo bip! tiene 143 filas, una por estación y línea. La
diferencia no es un error. El problema empieza cuando se cruzan como si cada fila
representara lo mismo.

La conciliación usa dos criterios independientes —nombre normalizado y proximidad— y
convierte sus desacuerdos en casos para revisar. Esto descubre once estaciones del
archivo bip! a más de 500 metros de la ubicación GTFS y una separación de 26,2 km entre
dos filas que dicen «Plaza de Armas». Luego se responde la pregunta operativa: qué
estaciones tienen buses a 200 metros y cuántos servicios distintos las atienden.


## 2 · La solución que se va a aplicar

Se diagnostica el grano antes del cruce, se reducen las filas bip! a estaciones sin
ocultar su dispersión y se confronta nombre con distancia. Cuatro detectores de calidad
declaran qué comparan y cuál es su punto ciego. Solo después se calculan las paradas y
servicios de bus cercanos. La respuesta se repite con la otra fuente de coordenadas
para medir cuánto depende del proveedor elegido.


## Implementaciones comparables

La ruta principal conserva el caso completo en Pandas. El laboratorio comparable ejecuta Pandas y Polars en este runtime sobre el mismo CSV. R usa el mismo contrato y se valida en CI; el runtime Python estándar de Colab no ejecuta R de forma nativa.

#### Contrato común del ejercicio

- **Objetivo.** Comparar cuántos hallazgos aporta cada detector independiente de coordenadas.
- **Fuente que leen las tres implementaciones.**
  [`resultados/detectores.csv`](resultados/detectores.csv), CSV UTF-8 versionado y producido
  por la resolución completa del caso. Los datos originales, su publicador, fecha,
  licencia, URL y huella se documentan en [`case.json`](case.json) y, cuando existe,
  `procedencia.json`. Este CSV es un **resultado reproducible**, no se presenta como
  una nueva fuente oficial.
- **Alcance.** hallazgos por detector independiente. Los tres scripts comparables no sustituyen la adquisición,
  limpieza ni controles completos de los pasos Pandas: aíslan el indicador para que
  una diferencia observada sea atribuible al motor.
- **Fórmula o regla.** hallazgos_d = Σ 1(detector d marca la fila). Los detectores no son mutuamente excluyentes, por lo que sus barras no deben sumarse como un total.
- **Interpretación.** La diferencia entre barras muestra puntos ciegos del control, no una competencia entre algoritmos.
- **Error frecuente.** Sumar detectores duplica estaciones marcadas por más de una regla; use el identificador de estación para una unión.

| Campo | Tipo observado | Función en el ejercicio |
| --- | --- | --- |
| `detector` | `str` | categoría/eje X |
| `compara` | `str` | campo auxiliar del contrato |
| `fuente_externa` | `str` | campo auxiliar del contrato |
| `hallazgos` | `int64` | medida/eje Y |
| `unidad` | `str` | campo auxiliar del contrato |
| `ciego_a` | `str` | campo auxiliar del contrato |

#### Qué cambia entre motores

| Dimensión | Pandas | Polars | R (`dplyr`/`readr`) |
| --- | --- | --- | --- |
| Modelo | DataFrame mutable por operaciones | expresiones columnares y plan optimizable | tubería declarativa sobre `tibble` |
| Nulos | `NaN`/`NA`; `groupby` exige decidir `dropna` | `null` separado de `NaN` | `NA`; muchas funciones exigen `na.rm` explícito |
| Orden | varias operaciones lo conservan, pero se ordena explícitamente | no se presupone tras agrupar | se fija con `arrange` |
| Gráfico | Matplotlib | Matplotlib tras extraer sólo X/Y | ggplot2 y su gramática de capas |
| Ventaja típica | ecosistema amplio y aprendizaje gradual | paralelismo, tipos y consultas grandes | estadística, comunicación y análisis reproducible |
| Límite aquí | el índice puede introducir alineaciones implícitas | añade una dependencia y otra semántica de tipos | requiere runtime y paquetes R independientes |

La equivalencia se valida sobre CSV normalizados con tolerancia numérica `1e-9`; no
se exige que la sintaxis ni los píxeles sean idénticos. Para ejecutar en un entorno
limpio consulte [`docs/ejecucion-portable.md`](../../docs/ejecucion-portable.md).

### Pandas + Matplotlib

```python
"""Laboratorio comparable Pandas + Matplotlib del caso correspondencia_metro.

Lee el mismo contrato CSV que las versiones Polars y R. La resolución completa del
caso permanece en ``solution.py`` y en los pasos del cuaderno; este archivo aísla el
indicador central para que la comparación entre motores sea controlada.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Buscar la raíz desde el propio archivo evita rutas C:\... específicas del autor y
# funciona igual después de clonar el repositorio en Linux, Windows o Google Colab.
ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "correspondencia_metro" / "resultados" / "detectores.csv"
REQUIRED = {"detector", "hallazgos"}


def analizar() -> pd.DataFrame:
    """Carga, valida y transforma el indicador; no modifica el archivo de entrada."""
    datos = pd.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # La expresión conserva explícitos filtro, agrupación, orden y selección para que
    # el estudiante pueda alinearlos con las expresiones de Polars y dplyr.
    resultado = datos.sort_values("hallazgos", ascending=False)[["detector", "hallazgos"]]
    return resultado.reset_index(drop=True)


def graficar(resultado: pd.DataFrame, destino: Path) -> Path:
    """Representa exactamente las columnas comparadas y explicita sus unidades."""
    x = resultado["detector"].tolist()
    y = resultado["hallazgos"].tolist()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh([str(value) for value in x], y, color="#11253B")
    ax.invert_yaxis()  # el primer puesto del ranking queda arriba, como se lee una tabla
    ax.set_xlabel("Hallazgos (recuento)")
    ax.set_ylabel("Detector")
    ax.set_xlim(left=0)
    ax.set_title("Cada detector encuentra un punto ciego distinto")
    ax.grid(axis="y", alpha=0.25)
    fig.text(0.01, 0.01, "Fuente: Resultados reproducibles del caso P03/P10 · Período: corte publicado", fontsize=8, color="#58677A")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado.to_string(index=False))
    if args.salida:
        resultado.to_csv(args.salida, index=False)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `python cases/correspondencia_metro/cuaderno/solucion_pandas.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/correspondencia_metro/cuaderno/solucion_pandas.py', '--grafico', 'solucion_pandas-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_pandas-colab.png'))


### Polars + Matplotlib

```python
"""Laboratorio comparable Polars + Matplotlib del caso correspondencia_metro.

Usa el mismo CSV, columnas, reglas y escala que Pandas y R. Polars expresa las
transformaciones como un plan de expresiones columnares; esa diferencia de método es
el objeto de aprendizaje, no una razón para cambiar la pregunta.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "correspondencia_metro" / "resultados" / "detectores.csv"
REQUIRED = {"detector", "hallazgos"}


def analizar() -> pl.DataFrame:
    """Carga y valida antes de transformar para producir un error explicativo."""
    datos = pl.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # select al final fija el contrato de salida y evita comparar columnas auxiliares.
    return datos.sort("hallazgos", descending=True).select(["detector", "hallazgos"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    """Convierte sólo las series del gráfico; el análisis permanece en Polars."""
    x = resultado["detector"].to_list()
    y = resultado["hallazgos"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh([str(value) for value in x], y, color="#11253B")
    ax.invert_yaxis()  # el primer puesto del ranking queda arriba, como se lee una tabla
    ax.set_xlabel("Hallazgos (recuento)")
    ax.set_ylabel("Detector")
    ax.set_xlim(left=0)
    ax.set_title("Cada detector encuentra un punto ciego distinto")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultados reproducibles del caso P03/P10 · Período: corte publicado",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.salida:
        resultado.write_csv(args.salida)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/correspondencia_metro/cuaderno/solucion_polars.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/correspondencia_metro/cuaderno/solucion_polars.py', '--grafico', 'solucion_polars-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_polars-colab.png'))


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Laboratorio comparable R + dplyr/readr + ggplot2 del caso correspondencia_metro.
#
# La resolución completa sigue en solution.py y en los pasos Pandas. Este script
# cambia el motor sobre el mismo contrato CSV para estudiar sintaxis y semántica sin
# cambiar datos, indicador ni escala gráfica.

required <- c("readr", "dplyr")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  stop(
    "Faltan paquetes: ", paste(missing, collapse = ", "),
    ". Instálelos con install.packages(c('readr', 'dplyr', 'ggplot2'))."
  )
}

# Buscar pyproject.toml hacia arriba hace portable el script entre terminal, RStudio,
# Posit Cloud, IRkernel y un clon en Colab. LAB_ROOT permite una ruta explícita sin
# codificar la unidad o el directorio del desarrollador.
find_root <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "pyproject.toml"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("No se encontró la raíz del repositorio.")
    current <- parent
  }
}
root <- Sys.getenv("LAB_ROOT", unset = find_root())
source_csv <- file.path(root, "cases", "correspondencia_metro", "resultados", "detectores.csv")

# readr conserva nombres y tipos previsibles. La validación temprana produce un error
# que explica el contrato roto, en vez de fallar más tarde dentro de una gráfica.
datos <- readr::read_csv(source_csv, show_col_types = FALSE, progress = FALSE)
required_columns <- c("detector", "hallazgos")
missing_columns <- setdiff(required_columns, names(datos))
if (length(missing_columns)) stop("Faltan columnas: ", paste(missing_columns, collapse = ", "))

# dplyr expresa la misma selección, filtro, agrupación y orden que las otras vías.
resultado <- datos |> dplyr::arrange(dplyr::desc(.data$hallazgos)) |> dplyr::select(.data$detector, .data$hallazgos)
print(resultado)

args <- commandArgs(trailingOnly = TRUE)
value_after <- function(flag) {
  pos <- match(flag, args)
  if (is.na(pos)) return(NULL)
  if (pos == length(args)) stop(flag, " necesita una ruta de salida")
  args[pos + 1]
}

csv_output <- value_after("--salida")
if (!is.null(csv_output)) readr::write_csv(resultado, csv_output)

plot_output <- value_after("--grafico")
if (!is.null(plot_output)) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["detector"]], y = .data[["hallazgos"]])
  ) +
    ggplot2::geom_col(fill = "#11253B") +
    ggplot2::scale_x_discrete(limits = rev) +
    ggplot2::coord_flip() +
    ggplot2::labs(
      title = "Cada detector encuentra un punto ciego distinto",
      x = "Hallazgos (recuento)", y = "Detector",
      caption = "Fuente: Resultados reproducibles del caso P03/P10 · Período: corte publicado"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(plot_output, grafico, width = 9, height = 5, dpi = 150)
  cat(plot_output, "\n")
}
```

Ejecutar con `Rscript cases/correspondencia_metro/cuaderno/solucion.R`.


### Ejecutar R

Use `solucion.R` desde R/RStudio, Posit Cloud o Jupyter con IRkernel. También se incluye `solucion.Rmd`. En Colab, seleccione un runtime R compatible o use la instalación experimental documentada; se marca como compatibilidad documentada, no como ejecución validada.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.correspondencia_metro import solution as p10

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Diagnosticar el grano de las dos fuentes


126 y 143 se reconcilian explicando qué representa cada fila.

> ⚠️ Forzar una unión uno a uno elimina 17 filas legítimas de estaciones de combinación.


In [ ]:
resultados = p10.analizar()
resultados['grano']


### Paso 2 · Conciliar por nombre y por distancia


Dos criterios independientes permiten medir cobertura y desacuerdo.

> ⚠️ El nombre encuentra 125 de 126; la fila restante no debe desaparecer en silencio.


In [ ]:
resultados['conciliacion']


### Paso 3 · Revisar donde los criterios discrepan


Los trece desacuerdos son el conjunto pequeño que requiere inspección.

> ⚠️ Elegir automáticamente el criterio con más coincidencias puede confirmar una coordenada corrupta.


In [ ]:
resultados['criterios_discrepantes']


### Paso 4 · Declarar el punto ciego de cada detector


Calidad significa saber qué error puede y qué error no puede observar cada control.

> ⚠️ UTM y lat/lon pueden estar ambas equivocadas de la misma forma: el detector interno no lo ve.


In [ ]:
resultados['detectores']


### Paso 5 · Medir la correspondencia a 200 metros


Cada estación se cruza con paradas de bus por distancia haversine.

**En Excel:** `=CONTAR.SI.CONJUNTO(estacion;A2;distancia_m;"<=200")`

Debe dar: **126 de 126 estaciones con al menos una parada**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Contar filas de horarios multiplica un mismo servicio; la unidad es servicio distinto.


In [ ]:
resultados['correspondencia']


### Paso 6 · Mostrar la distribución de servicios


La mediana, el p90 y el máximo evitan resumir toda la red con un único extremo.

> ⚠️ El máximo de 55 describe Santa Lucía, no la estación típica; la mediana es 13.


In [ ]:
resultados['resumen']


### Paso 7 · Identificar las estaciones con más alternativas


La tabla prioriza estaciones por servicios distintos y conserva el número de paradas.

> ⚠️ Más paradas físicas no implica necesariamente más servicios diferentes.


In [ ]:
resultados['mejores_correspondencias']


### Paso 8 · Medir la sensibilidad a la fuente de coordenadas


La misma pregunta se recalcula con GTFS y bip! para cuantificar dependencia.

> ⚠️ Con bip!, tres estaciones parecerían no tener bus y 48 cambian su conteo de servicios.


In [ ]:
resultados['sensibilidad_a_la_fuente']


### Paso 9 · Separar hallazgo de explicación


El cierre publica evidencia y límites sin atribuir causas no observadas.

> ⚠️ Un desvío demuestra inconsistencia entre fuentes, no cuál de las dos es correcta ni por qué.


In [ ]:
resultados['detector_4_contra_gtfs']


## 4 · El resultado

Las 126 estaciones GTFS tienen al menos una parada de bus a 200 m y la mediana es 13 servicios. Pero 48 estaciones cambian su conteo al usar la coordenada bip!: la fuente es parte del método.

## 5 · Qué queda aprendido

1. Dos conteos distintos pueden ser correctos si sus filas tienen granos diferentes.
2. El desacuerdo entre criterios es información para revisar, no ruido que deba borrarse.
3. Toda prueba de calidad tiene un punto ciego medible.
4. Una inconsistencia entre fuentes no autoriza a declarar cuál es la verdadera.
